# Final Jewelry RAG generation evaluation

Generates answers for the unified text and image gold cases using full-corpus retrieval. Gold labels are excluded from retrieval and generation inputs and used only by the post-generation evaluator. Per-case answers and scores are checkpointed so the notebook can resume after API limits or interruptions.

## 1. Setup and load gold cases

In [ ]:
from pathlib import Path
import importlib, json, os, re, sys, time, unicodedata
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents, cwd / "ai-service"] if (p / "src" / "rag" / "scripts").is_dir() and (p / "rag_documents").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the Jewelry project or ai-service directory.")
load_dotenv(PROJECT_ROOT / ".env", override=True)
RAG_SRC = PROJECT_ROOT / "src" / "rag"
if str(RAG_SRC) not in sys.path:
    sys.path.insert(0, str(RAG_SRC))

embedding_module = importlib.import_module("scripts.5_embeddings")
retrieval = importlib.import_module("scripts.7_retrieval")
image_embeddings = importlib.import_module("scripts.10_image_embeddings")
generation = importlib.import_module("scripts.11_generation")
EVALUATION_DIR = PROJECT_ROOT / "data" / "rag" / "evaluations"
GOLD_FILE = EVALUATION_DIR / "final_retrieval_generation_gold.json"
RESULTS_FILE = EVALUATION_DIR / "final_generation_evaluations.json"
gold = json.loads(GOLD_FILE.read_text(encoding="utf-8"))
CASES = gold["cases"]
MODEL_NAME = "gemini-embedding-2"
IMAGE_MODEL_NAME = image_embeddings.DEFAULT_CONTEXT_MODEL
GENERATOR_MODEL = generation.DEFAULT_LLM_MODEL
JUDGE_MODEL = os.getenv("GENERATION_JUDGE_MODEL", "gemini-3.6-flash")
TOP_K = 5
CACHE_ROOT = PROJECT_ROOT / "data" / "rag" / "embeddings"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "rag" / "processed_documents"
RESET_CHECKPOINT = False  # Set True only to intentionally discard saved progress.
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
judge_client = generation.get_llm_client()
print("Cases:",len(CASES),"generator:",GENERATOR_MODEL,"judge:",JUDGE_MODEL)
print("Text/image embedding vectors will load from cache only; Gemini is called only for generation and judging.")
print("Checkpoint:",RESULTS_FILE)


## 2. Load cached text and image embeddings

In [ ]:
chunk_records=[]
for path in sorted(PROCESSED_ROOT.glob("*/chunks.json")):
    chunk_records.extend(json.loads(path.read_text(encoding="utf-8")).get("chunks",[]))
chunks_by_id=pd.DataFrame(chunk_records).drop_duplicates("chunk_id").set_index("chunk_id")
chunk_order_file=CACHE_ROOT/"BAAI__bge-m3"/"chunks.json"
if not chunk_order_file.is_file(): raise FileNotFoundError("Saved chunk-order index missing; run notebook 4 first.")
chunk_ids=json.loads(chunk_order_file.read_text(encoding="utf-8"))["chunk_ids"]
missing_ids=[chunk_id for chunk_id in chunk_ids if chunk_id not in chunks_by_id.index]
if missing_ids: raise ValueError(f"Chunk-order index references missing chunks: {missing_ids[:5]}")
chunks=chunks_by_id.loc[chunk_ids].reset_index()
CURRENT_PDF_NAMES={path.name for path in (PROJECT_ROOT/"rag_documents").rglob("*.pdf")}
chunks=chunks[chunks["file_name"].isin(CURRENT_PDF_NAMES)].reset_index(drop=True)
if chunks.empty: raise FileNotFoundError("No saved text chunks for current source PDFs. Run notebooks 1-4 first.")
chunk_pack=embedding_module.load_cached_gemini_embeddings(chunks["text"].astype(str).tolist(),
    text_type="passage",model_name=MODEL_NAME,cache_root=CACHE_ROOT)
chunk_vectors=chunk_pack["embeddings"]
text_cases=[case for case in CASES if "text" in case["modalities"]]
image_cases=[case for case in CASES if "image" in case["modalities"]]
text_query_pack=embedding_module.load_cached_gemini_embeddings([case["query"] for case in text_cases],
    text_type="query",model_name=MODEL_NAME,cache_root=CACHE_ROOT)
query_vectors={case["query"]:vector for case,vector in zip(text_cases,text_query_pack["embeddings"])}
image_query_cache=PROJECT_ROOT/"data"/"rag"/"embeddings"/"image_query_embeddings"
for case in image_cases:
    pack=embedding_module.load_cached_gemini_embeddings([case["query"]],text_type="query",
        model_name=IMAGE_MODEL_NAME,cache_root=image_query_cache)
    query_vectors[case["query"]]=pack["embeddings"][0]
class CachedQueryModel:
    def encode(self,sentences,**kwargs): return np.asarray([query_vectors[str(s).strip()] for s in sentences],dtype=np.float32)
query_model=CachedQueryModel()
image_records=image_embeddings.load_image_records(processed_root=PROCESSED_ROOT)
image_pack=image_embeddings.load_cached_image_embeddings(image_records,
    output_root=PROJECT_ROOT/"data"/"rag"/"embeddings"/"image_vlm_descriptions",
    context_model_name=IMAGE_MODEL_NAME)
image_query_model=query_model
images_by_id={str(r["image_id"]):r for r in image_records}
print("Current source PDFs:",len(CURRENT_PDF_NAMES),"text chunks:",len(chunks),"image records:",len(image_records))
print("Loaded cached text passage/query vectors and per-case image query vectors; no embedding API calls.")


## 3. Retrieval, post-generation judge, and checkpoint helpers

In [ ]:
def retrieve_evidence(query):
    # Full-corpus retrieval only. No gold PDF/image/keyword labels enter these calls.
    text_frame=retrieval.retrieve_dense(query,query_model,chunk_vectors,chunks,top_k=TOP_K,model_name=MODEL_NAME)
    image_frame=retrieval.retrieve_images(query,method="context_dense",model=image_query_model,
        model_name=IMAGE_MODEL_NAME,top_k=TOP_K,context_embeddings=image_pack["context_embeddings"],image_records=image_records)
    text_contexts=text_frame.to_dict(orient="records")
    visual_contexts=[]
    for row in image_frame.to_dict(orient="records"):
        record=images_by_id.get(str(row.get("image_id")),{})
        visual_contexts.append({**row,"visual_description":record.get("visual_description","")})
    return text_contexts,visual_contexts

def normalized(value):
    value=unicodedata.normalize("NFKD",str(value).casefold())
    return "".join(char for char in value if char.isalnum())

def evaluate_generation_answer(case,answer,evidence):
    reference=case.get("generation_reference",{})
    prompt={"question":case["query"],"answer":answer,"retrieved_evidence":evidence,"gold_reference":reference}
    messages=[
      {"role":"system","content":"You are an independent evaluator for a Jewelry RAG answer. Score the answer against the question, retrieved evidence, and gold reference. Gold labels are for evaluation only. Award faithfulness for claims supported by retrieved evidence; correctness for agreement with the gold concepts; relevance for directly answering the question. Do not require every reference detail if the evidence retrieved does not contain it. Return only JSON with integer scores 1-4 for faithfulness, correctness, relevance, plus a brief note."},
      {"role":"user","content":json.dumps(prompt,ensure_ascii=False)}]
    response=judge_client.chat.completions.create(model=JUDGE_MODEL,messages=messages,temperature=0,max_tokens=400)
    raw=(response.choices[0].message.content or "").strip()
    match=re.search(r"\{.*\}",raw,re.S)
    if not match: raise ValueError(f"Judge did not return JSON: {raw[:300]}")
    scores=json.loads(match.group())
    for name in ("faithfulness","correctness","relevance"):
        scores[name]=max(1,min(4,int(scores[name])))
    keywords=case.get("expected_keywords",[])
    norm_answer=normalized(answer)
    scores["keyword_coverage"]=(sum(1 for kw in keywords if normalized(kw) in norm_answer)/len(keywords)) if keywords else None
    scores["note"]=str(scores.get("note", ""))[:500]
    return scores

def save_progress(rows,complete=False):
    payload={"schema_version":1,"gold_dataset":str(GOLD_FILE.relative_to(PROJECT_ROOT)),
        "configuration":{"text_retrieval":"dense","text_embedding_model":MODEL_NAME,"image_retrieval":"context_dense",
            "image_embedding_model":IMAGE_MODEL_NAME,"top_k":TOP_K,"generator_model":GENERATOR_MODEL,"judge_model":JUDGE_MODEL},
        "n_expected":len(CASES),"n_generated":sum(bool(r.get("answer")) for r in rows),
        "n_evaluated":sum(bool(r.get("evaluation")) for r in rows),"complete":complete,"results":rows}
    RESULTS_FILE.write_text(json.dumps(payload,ensure_ascii=False,indent=2,default=str),encoding="utf-8")


## 4. Generate and evaluate (rerun this cell to resume)

In [ ]:
if RESET_CHECKPOINT and RESULTS_FILE.exists():
    RESULTS_FILE.unlink()
rows=[]
if RESULTS_FILE.exists():
    try: rows=json.loads(RESULTS_FILE.read_text(encoding="utf-8")).get("results",[])
    except Exception as exc: print("Could not load checkpoint; starting fresh:",exc)
by_id={row.get("id"):row for row in rows if row.get("id")}
print(f"Resuming with {sum(bool(r.get('answer')) for r in by_id.values())}/{len(CASES)} generated and {sum(bool(r.get('evaluation')) for r in by_id.values())}/{len(CASES)} judged")

for index,case in enumerate(CASES,1):
    row=by_id.get(case["id"])
    try:
        if row is None or not row.get("answer"):
            print(f"[{index}/{len(CASES)}] Generating {case['id']} ({case['language']})",flush=True)
            text_contexts,image_contexts=retrieve_evidence(case["query"])
            answer=generation.generate_answer(case["query"],text_contexts,image_contexts=image_contexts,model=GENERATOR_MODEL)
            row={"id":case["id"],"language":case["language"],"modalities":case["modalities"],
                 "cross_pdf":case.get("cross_pdf",False),"query":case["query"],"answer":answer,
                 "text_sources":[{"file_name":r.get("file_name"),"page_number":r.get("page_number"),"chunk_id":r.get("chunk_id")} for r in text_contexts],
                 "image_sources":[{"image_id":r.get("image_id"),"file_name":r.get("file_name")} for r in image_contexts],
                 "retrieval_counts":{"text":len(text_contexts),"images":len(image_contexts)},
                 "evidence_for_judge":{"text":text_contexts,"images":[{k:v for k,v in r.items() if k in {"image_id","file_name","page_numbers","visual_description"}} for r in image_contexts]},
                 "evaluation":None}
            by_id[case["id"]]=row
            rows=[by_id[c["id"]] for c in CASES if c["id"] in by_id]
            save_progress(rows)
        if not row.get("evaluation"):
            print(f"[{index}/{len(CASES)}] Judging {case['id']}",flush=True)
            # Reference labels are consulted only after answer generation is complete.
            row["evaluation"]=evaluate_generation_answer(case,row["answer"],row["evidence_for_judge"])
            by_id[case["id"]]=row
            rows=[by_id[c["id"]] for c in CASES if c["id"] in by_id]
            save_progress(rows)
    except Exception as exc:
        print(f"Stopped at {case['id']}: {type(exc).__name__}: {exc}",flush=True)
        print("Progress is saved. Re-run this cell to resume.")
        break
rows=[by_id[c["id"]] for c in CASES if c["id"] in by_id]
print(f"Checkpoint now has {len(rows)}/{len(CASES)} cases.")


## 5. Review generation metrics

In [ ]:
df=pd.DataFrame(rows)
scored=df[df["evaluation"].apply(lambda value:isinstance(value,dict))].copy() if len(df) else pd.DataFrame()
if len(scored):
    evaluation_frame=pd.json_normalize(scored["evaluation"])
    scored=pd.concat([scored.reset_index(drop=True),evaluation_frame.reset_index(drop=True)],axis=1)
    def report(group):
        return {"cases":int(len(group)),"faithfulness_1_to_4":float(group.faithfulness.mean()),
            "correctness_1_to_4":float(group.correctness.mean()),"relevance_1_to_4":float(group.relevance.mean()),
            "mean_keyword_coverage":float(group.keyword_coverage.dropna().mean()) if group.keyword_coverage.notna().any() else None}
    overall=report(scored)
    by_language={lang:report(group) for lang,group in scored.groupby("language")}
    by_modality={modality:report(scored[scored.modalities.apply(lambda values:modality in values)]) for modality in ["text","image"]}
    cross=report(scored[scored.cross_pdf]) if scored.cross_pdf.any() else None
    print(json.dumps({"overall":overall,"by_language":by_language,"by_modality":by_modality,"cross_pdf":cross,
        "complete":len(scored)==len(CASES)},ensure_ascii=False,indent=2))
    display(scored[["id","language","modalities","faithfulness","correctness","relevance","keyword_coverage"]].head(20))
    save_progress(rows,complete=len(scored)==len(CASES))
else:
    print("No fully evaluated cases yet. Run the generation cell above.")
